## Great, now that we discussed a little let's continue

Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.

Please explain, how we will achieve reproducibility for the course  with this approach.


Using nextflow pipelines that follow the nf-core guidelines, which ensure reproducibility by packaging, good documentation and CI testing. 

You have successfully downloaded 2 of the fastq files we will use in our study.

What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

The first steps are preprocessing:  
-Merge re-sequenced FastQ files (cat)  
-Sub-sample FastQ files and auto-infer strandedness (fq, Salmon)  
-Read QC (FastQC)  
-UMI extraction (UMI-tools)  
-Adapter and quality trimming (Trim Galore or fastp)  
-Deduplication (SAMtools dedup, UMI-tools)
-Removal of genome contaminants (BBSplit)  
-Removal of ribosomal RNA (SortMeRNA)    
-Creation of a feature matrix

and then afterwards alignment using HiSAT2, then they can be sorted and indexed using Samtools, the transcript can be assembled and then the Quality Control can be done based on different packages (for example DeSeq2, which was used in the paper). 


Analyze the 2 files using an nf-core pipeline.

What does this pipeline do?

Which are the main tools that will be used in the pipeline?

As described the pipeline that will be used is rna-seq. It aligns and quality controls the inputted .fastq files. The main tools are already described in detail for the question above.

As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.

Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.

Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

In [3]:
import pandas as pd
#prepare samplesheet -> already produced by the nf-core fetchngs pipeline, so just read in and add strandedness -> TruSeq mRNA library prep kit is reverse stranded
samplesheet = pd.read_csv("/Users/peterbrederlow/Documents/Uni/MasterBioinformatik/SS25/WorkflowCourse/computational-workflows-2025/SRFetch_results/samplesheet/samplesheet.csv")
samplesheet["strandedness"] = "reverse"
samplesheet.to_csv("/Users/peterbrederlow/Documents/Uni/MasterBioinformatik/SS25/WorkflowCourse/computational-workflows-2025/SRFetch_results/samplesheet/samplesheet_fixed.csv", index=False)
print(samplesheet.head())


        sample                                            fastq_1  \
0  SRX19144486  ./SRFetch_results/fastq/SRX19144486_SRR2319551...   
1  SRX19144488  ./SRFetch_results/fastq/SRX19144488_SRR2319551...   

                                             fastq_2 run_accession  \
0  ./SRFetch_results/fastq/SRX19144486_SRR2319551...   SRR23195516   
1  ./SRFetch_results/fastq/SRX19144488_SRR2319551...   SRR23195511   

  experiment_accession sample_accession secondary_sample_accession  \
0          SRX19144486     SAMN32880530                SRS16557078   
1          SRX19144488     SAMN32880528                SRS16557080   

  study_accession secondary_study_accession submission_accession  ...  \
0     PRJNA926667                 SRP418759           SRA1578893  ...   
1     PRJNA926667                 SRP418759           SRA1578893  ...   

  sample_title                                   experiment_title  \
0           H2  Illumina HiSeq 2000 sequencing: GSM6958543: H2...   
1           

In [ ]:
# post here the command you used to run nf-core/rnaseq
nextflow run nf-core/rnaseq \
    --input "/Users/peterbrederlow/Documents/Uni/MasterBioinformatik/SS25/WorkflowCourse/computational-workflows-2025/SRFetch_results/samplesheet/samplesheet_fixed.csv" \
    --outdir "./rnaseq_results" \
    --genome Grch38 \
    -profile docker

Explain all the parameters you set and why you set them in this way.



The samplesheet has the specified names of the fastq_files and other metadata that is necessary to preprocess and align the data. The genome specified is the reference genome to align against and the outdir and profile is similar to all other nextflow elements.  
Unfortunately the pipeline does not run trough due to CPU/GPU limitations, hence the results cannot be browsed. 

## Browsing the results

How did the pipeline perform?

Unfortunately CPU/RAM limitations prevented the pipeline from running through on my laptop, although generally speaking the executor process limits can be limited in the nextflow.config file of the pipeline. This unfortuantely did not work for me either, but could have been another fix.

Explain the quality control steps. Are you happy with the quality and why. If not, why not.
Please give additional information on : 
- ribosomal rRNA
- Duplication
- GC content

What are the possible steps that could lead to poorer results?

Quality control steps typically include assessing read quality, adapter contamination, duplication levels, GC content, and the presence of ribosomal rRNA. High-quality data should have minimal adapter contamination, low duplication rates, and GC content within the expected range for the organism. Ribosomal rRNA contamination should be low in non-rRNA-focused experiments. Poor results can arise from issues like degraded samples, contamination, sequencing errors, or improper library preparation.
GC content measures the percentage of guanine and cytosine bases, with deviations indicating PCR bias, sequencing artifacts, or contamination.
Duplication on the other hand refers to identical reads, with high levels reducing library complexity and potentially biasing results. Although duplicated reads are also somewhat expected in RNA sequencing counts with several transcripts, this needs to be accounted for as well. 

Would you exclude any samples? If yes, which and why?

As already described, the pipeline did not run through leaving no way to inspect the fastqc or multiqc report.

What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

Additional steps woul be to analyze the differential expression of the genes, using either the differential abundance pipeline from nextflow or well known tools such as DESeq2 in R. Afterwards the differentially expressed genes could be used to calculate a gene set enrichment analysis (GSEA) or o further eploratory analysis.